# Notebook Overview — Evaluate Development Results

## Purpose

This notebook evaluates development-subset VideoQA results generated by previous project notebooks and produces quantitative performance metrics, analysis summaries, visualizations, and reporting artifacts.

Because NExT-QA is a multiple-choice VideoQA benchmark, multiple-choice accuracy is the primary evaluation metric when experiment outputs use `answer_mode = "multiple_choice"`. Exact-match and partial-match text comparisons are retained as secondary diagnostic metrics that provide additional insight into model-generated responses but are not considered the primary benchmark score.

During the current development phase, this notebook evaluates baseline VideoQA results generated by Notebook 01. The notebook is designed to support future comparative evaluation of baseline, pretrained-representation, and autoencoder-based VideoQA experiments using a common evaluation framework.

Evaluation procedures include prediction validation, multiple-choice accuracy analysis, reasoning-category analysis, question-type analysis, runtime analysis, visualization generation, and experiment reporting.

## Inputs

* VideoQA prediction results
* Experiment summary reports
* Runtime statistics
* NExT-QA validation annotations
* Project configuration settings

## Outputs

* Multiple-choice evaluation metrics
* Prediction verification summaries
* Choice prediction accuracy summaries
* Reasoning-category performance analyses
* Question-type performance analyses
* Answer-length analyses
* Runtime analyses
* Performance visualizations
* Evaluation reports
* Saved reporting artifacts

## Workflow

The workflow begins by loading experiment outputs and reference annotation data. Input files and required prediction columns are validated before evaluation datasets are prepared.

Prediction results are matched with NExT-QA annotation records to support reasoning-category and question-type analysis. For multiple-choice experiments, predicted answer choices are compared against ground-truth answer choices to compute benchmark accuracy metrics. Exact-match and partial-match text metrics are also computed as secondary diagnostic measures.

Evaluation metrics, answer-length statistics, runtime summaries, category-level analyses, and visualization artifacts are generated and saved for later reporting and experiment comparison.

The notebook provides a common evaluation framework that can be applied consistently across baseline, pretrained-representation, and autoencoder-based VideoQA experiments.

## Notes

This notebook does not perform VideoQA inference and does not require access to raw video files. Evaluation is performed using saved experiment outputs and NExT-QA reference annotations.

For NExT-QA multiple-choice experiments, choice accuracy is the primary benchmark metric. Exact-match and partial-match text comparisons are retained for diagnostic analysis only. Future work may incorporate semantic similarity metrics and additional benchmark measures to complement the current evaluation framework.


### 🔷 Step 1 — Initialize Evaluation Environment

* Clone the project repository using sparse checkout to minimize download size and runtime initialization overhead.
* Authenticate access to the private GitHub repository using a fine-grained access token stored in Google Colab Secrets.
* Configure the local notebook workspace and change to the repository working directory.
* Verify that required repository files, configuration modules, and dataset resources are available for evaluation.
* Prepare the notebook environment for loading saved experiment results and evaluation reference data.
* Optionally display repository paths, directory contents, and cloned files when `VERBOSE=True`.


In [ ]:
# ============================================================
# Step 1: Initialize Environment
# ============================================================

VERBOSE = True
REQUIRE_L4_GPU = False

import os
from pathlib import Path

import pandas as pd

from google.colab import userdata, drive

print("Initializing Notebook 08 environment...")
print("-" * 60)

# ------------------------------------------------------------
# Clone Required Repository Files
# ------------------------------------------------------------

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(
    REPO_BASE_DIR,
    REPO_NAME,
)

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets."
    )

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

os.chdir(REPO_BASE_DIR)

if not os.path.exists(REPO_DIR):

    print("Cloning project repository...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    os.chdir(REPO_DIR)

    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main

else:

    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):

    print("\nMounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)

else:

    print("\nGoogle Drive already mounted.")

# ------------------------------------------------------------
# Load Project Configuration and Utility Modules
# ------------------------------------------------------------

print("\nLoading project configuration...")

from src.videoqa_representation_config import *
from src.nextqa_metadata import *

required_paths = [
    Path("src"),
    QUESTIONS_DIR,
    METADATA_DIR,
    GOOGLE_DRIVE_ROOT,
]

missing_paths = [
    path
    for path in required_paths
    if not Path(path).exists()
]

if missing_paths:

    for path in missing_paths:
        print(f"Missing required path: {path}")

    raise FileNotFoundError(
        "One or more required project paths are missing."
    )

OUTPUTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EVALUATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Configuration loaded.")
print("Project paths initialized.")

# ------------------------------------------------------------
# Load NExT-QA Annotation Metadata
# ------------------------------------------------------------

print("\nLoading NExT-QA annotation metadata...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

print("\nDataset metadata ready.")
print(f"Annotation records : {len(annotations_df):,}")

if VERBOSE:

    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nNotebook 08 initialization complete.")
print("-" * 60)
print("Ready to evaluate VideoQA prediction artifacts.")



### 🔷 Step 2 — Restore and Load Evaluation Data

* Restore the configured VideoQA prediction artifacts from Google Drive to the local workspace.
* Load prediction records, prediction validation records, and experiment summary metadata produced by the upstream VideoQA notebook.
* Load NExT-QA annotations for the configured evaluation split.
* Verify that all required evaluation input artifacts are present, readable, and conform to the expected schema.
* Confirm that prediction records, validation records, summary metadata, and annotation records contain the required columns.
* Display dataset sizes, artifact locations, column information, and sample records to verify successful restoration and loading.

In [ ]:
# ============================================================
# Step 2: Restore and Load Evaluation Data
# ============================================================

import shutil
import pandas as pd

from src.videoqa_representation_config import *

print("Loading evaluation data...\n")

# ------------------------------------------------------------
# Evaluation Reference Files
# ------------------------------------------------------------

VALIDATION_ANNOTATIONS_CSV = (
    QUESTIONS_DIR / f"{EVALUATION_SPLIT}.csv"
)

# ------------------------------------------------------------
# Restore Evaluation Artifacts from Google Drive
# ------------------------------------------------------------

print("Restoring evaluation artifacts from Google Drive...")
print("-" * 60)

artifact_restore_pairs = [
    (
        EVALUATION_PREDICTIONS_DRIVE_CSV,
        EVALUATION_PREDICTIONS_CSV,
    ),
    (
        EVALUATION_VALIDATION_DRIVE_CSV,
        EVALUATION_VALIDATION_CSV,
    ),
    (
        EVALUATION_SUMMARY_DRIVE_CSV,
        EVALUATION_SUMMARY_CSV,
    ),
]

for drive_path, local_path in artifact_restore_pairs:

    if not drive_path.exists():
        raise FileNotFoundError(
            f"Required Google Drive artifact not found:\n"
            f"{drive_path}"
        )

    local_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        drive_path,
        local_path,
    )

    if not local_path.exists():
        raise FileNotFoundError(
            f"Failed to restore local artifact:\n"
            f"{local_path}"
        )

    print(f"Restored: {local_path.name}")

# ------------------------------------------------------------
# Verify Required Input Files
# ------------------------------------------------------------

required_input_files = [
    EVALUATION_PREDICTIONS_CSV,
    EVALUATION_VALIDATION_CSV,
    EVALUATION_SUMMARY_CSV,
    VALIDATION_ANNOTATIONS_CSV,
]

missing_input_files = [
    file_path
    for file_path in required_input_files
    if not file_path.exists()
]

if missing_input_files:

    for file_path in missing_input_files:
        print(f"Missing required input file: {file_path}")

    raise FileNotFoundError(
        "One or more required evaluation input files are missing."
    )

# ------------------------------------------------------------
# Load Evaluation Artifacts
# ------------------------------------------------------------

predictions_df = pd.read_csv(
    EVALUATION_PREDICTIONS_CSV
)

prediction_validation_df = pd.read_csv(
    EVALUATION_VALIDATION_CSV
)

prediction_summary_df = pd.read_csv(
    EVALUATION_SUMMARY_CSV
)

# ------------------------------------------------------------
# Load Evaluation Reference Data
# ------------------------------------------------------------

evaluation_annotations_df = pd.read_csv(
    VALIDATION_ANNOTATIONS_CSV
)

# ------------------------------------------------------------
# Validate Loaded Prediction Data
# ------------------------------------------------------------

required_prediction_columns = [
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
]

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in predictions_df.columns
]

if missing_prediction_columns:
    raise ValueError(
        "Prediction file is missing required columns: "
        f"{missing_prediction_columns}"
    )

# ------------------------------------------------------------
# Display Dataset Information
# ------------------------------------------------------------

print("\nLoaded Evaluation Data")
print("-" * 60)

print(
    f"Prediction records        : "
    f"{len(predictions_df):,}"
)

print(
    f"Validation records        : "
    f"{len(prediction_validation_df):,}"
)

print(
    f"Summary records           : "
    f"{len(prediction_summary_df):,}"
)

print(
    f"Annotation records        : "
    f"{len(evaluation_annotations_df):,}"
)

print("\nEvaluation Input Files")
print("-" * 60)

for file_path in required_input_files:
    print(file_path)

# ------------------------------------------------------------
# Display Available Columns
# ------------------------------------------------------------

print("\nPrediction Columns")
print("-" * 60)
print(list(predictions_df.columns))

print("\nValidation Columns")
print("-" * 60)
print(list(prediction_validation_df.columns))

print("\nSummary Columns")
print("-" * 60)
print(list(prediction_summary_df.columns))

print("\nAnnotation Columns")
print("-" * 60)
print(list(evaluation_annotations_df.columns))

# ------------------------------------------------------------
# Preview Loaded Data
# ------------------------------------------------------------

print("\nPrediction Preview")
display(predictions_df.head())

print("\nValidation Preview")
display(prediction_validation_df.head())

print("\nSummary Preview")
display(prediction_summary_df.head())

print("\nAnnotation Preview")
display(evaluation_annotations_df.head())



### 🔷 Step 3 — Prepare Evaluation Dataset

* Combine restored prediction records with NExT-QA annotation metadata for the configured evaluation split.
* Construct a unified evaluation dataset containing prediction results, ground-truth information, and supporting annotation metadata.
* Verify that the evaluation dataset contains all required columns and records.
* Validate dataset integrity by checking record counts, unique videos, and prediction coverage.
* Display summary statistics, available columns, and representative evaluation records prior to computing evaluation metrics.


In [ ]:
# ============================================================
# Step 3: Prepare Evaluation Dataset
# ============================================================

import pandas as pd

print("Preparing evaluation dataset...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

if "predictions_df" not in globals():
    raise NameError(
        "predictions_df was not found. Run Step 2 first."
    )

if "evaluation_annotations_df" not in globals():
    raise NameError(
        "evaluation_annotations_df was not found. Run Step 2 first."
    )

# ------------------------------------------------------------
# Filter annotations to evaluation records
# ------------------------------------------------------------

evaluation_reference_df = (
    evaluation_annotations_df
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Build evaluation dataset
# ------------------------------------------------------------

evaluation_dataset_df = (
    predictions_df
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Add annotation metadata when available
# ------------------------------------------------------------

annotation_metadata_columns = [
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    "qid",
    "type",
]

available_annotation_metadata_columns = [
    column
    for column in annotation_metadata_columns
    if column in evaluation_reference_df.columns
]

annotation_metadata_df = (
    evaluation_reference_df[
        available_annotation_metadata_columns
    ]
    .drop_duplicates()
)

evaluation_dataset_df = evaluation_dataset_df.merge(
    annotation_metadata_df,
    on=[
        VIDEO_ID_COLUMN,
        QUESTION_COLUMN,
    ],
    how="left",
    suffixes=(
        "",
        "_annotation",
    ),
)

# ------------------------------------------------------------
# Validate evaluation dataset
# ------------------------------------------------------------

required_evaluation_columns = [
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
]

missing_evaluation_columns = [
    column
    for column in required_evaluation_columns
    if column not in evaluation_dataset_df.columns
]

if missing_evaluation_columns:
    raise ValueError(
        "Evaluation dataset is missing required columns: "
        f"{missing_evaluation_columns}"
    )

if evaluation_dataset_df.empty:
    raise ValueError(
        "Evaluation dataset is empty."
    )

# ------------------------------------------------------------
# Display evaluation dataset summary
# ------------------------------------------------------------

print("Evaluation Dataset Prepared")
print("-" * 60)
print(f"Evaluation records        : {len(evaluation_dataset_df):,}")
print(f"Unique videos             : {evaluation_dataset_df[VIDEO_ID_COLUMN].nunique():,}")
print(f"Correct predictions       : {int(evaluation_dataset_df['choice_correct'].sum()):,}")

choice_accuracy = (
    evaluation_dataset_df["choice_correct"].mean()
)

print(f"Choice accuracy           : {choice_accuracy:.3f}")

if "type" in evaluation_dataset_df.columns:
    print(
        f"Question types            : "
        f"{evaluation_dataset_df['type'].nunique():,}"
    )

print("\nEvaluation Dataset Columns")
print("-" * 60)
print(list(evaluation_dataset_df.columns))

print("\nEvaluation Dataset Preview")
display(evaluation_dataset_df.head())



### 🔷 Step 4 — Verify Prediction Quality

* Validate prediction outputs against expected multiple-choice evaluation rules.
* Check for missing predictions, invalid predicted choices, and missing ground-truth choices.
* Confirm that predicted choices fall within the configured answer-choice set.
* Summarize prediction correctness, invalid prediction counts, and evaluation readiness.
* Display validation results before metric computation.


In [ ]:
# ============================================================
# Step 4: Verify Prediction Quality
# ============================================================

import pandas as pd

print("Verifying prediction quality...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

if "evaluation_dataset_df" not in globals():
    raise NameError(
        "evaluation_dataset_df was not found. Run Step 3 first."
    )

# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

required_quality_columns = [
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
]

missing_quality_columns = [
    column
    for column in required_quality_columns
    if column not in evaluation_dataset_df.columns
]

if missing_quality_columns:
    raise ValueError(
        "Evaluation dataset is missing required quality columns: "
        f"{missing_quality_columns}"
    )

# ------------------------------------------------------------
# Compute prediction quality checks
# ------------------------------------------------------------

valid_choice_values = set(range(len(CHOICE_COLUMNS)))

quality_checks = []

prediction_count = len(evaluation_dataset_df)

missing_ground_truth_count = int(
    evaluation_dataset_df["ground_truth_choice"].isna().sum()
)

missing_prediction_count = int(
    evaluation_dataset_df["predicted_choice"].isna().sum()
)

invalid_ground_truth_count = int(
    (
        ~evaluation_dataset_df["ground_truth_choice"]
        .dropna()
        .astype(int)
        .isin(valid_choice_values)
    ).sum()
)

invalid_prediction_count = int(
    (
        ~evaluation_dataset_df["predicted_choice"]
        .dropna()
        .astype(int)
        .isin(valid_choice_values)
    ).sum()
)

correct_prediction_count = int(
    evaluation_dataset_df["choice_correct"].sum()
)

incorrect_prediction_count = (
    prediction_count - correct_prediction_count
)

choice_accuracy = (
    correct_prediction_count / prediction_count
    if prediction_count > 0
    else 0.0
)

quality_checks.extend(
    [
        {
            "check": "prediction_records",
            "value": prediction_count,
        },
        {
            "check": "missing_ground_truth_choices",
            "value": missing_ground_truth_count,
        },
        {
            "check": "missing_predictions",
            "value": missing_prediction_count,
        },
        {
            "check": "invalid_ground_truth_choices",
            "value": invalid_ground_truth_count,
        },
        {
            "check": "invalid_predictions",
            "value": invalid_prediction_count,
        },
        {
            "check": "correct_predictions",
            "value": correct_prediction_count,
        },
        {
            "check": "incorrect_predictions",
            "value": incorrect_prediction_count,
        },
        {
            "check": "choice_accuracy",
            "value": choice_accuracy,
        },
    ]
)

prediction_quality_df = pd.DataFrame(
    quality_checks
)

# ------------------------------------------------------------
# Validate readiness
# ------------------------------------------------------------

blocking_issue_count = (
    missing_ground_truth_count
    + missing_prediction_count
    + invalid_ground_truth_count
    + invalid_prediction_count
)

evaluation_ready = blocking_issue_count == 0

if not evaluation_ready:
    raise ValueError(
        "Prediction quality checks failed. "
        "Resolve missing or invalid choices before evaluation."
    )

# ------------------------------------------------------------
# Display prediction quality summary
# ------------------------------------------------------------

print("Prediction Quality Verification")
print("-" * 60)
print(f"Prediction records        : {prediction_count:,}")
print(f"Correct predictions       : {correct_prediction_count:,}")
print(f"Incorrect predictions     : {incorrect_prediction_count:,}")
print(f"Choice accuracy           : {choice_accuracy:.3f}")
print(f"Evaluation ready          : {evaluation_ready}")

print("\nPrediction Quality Checks")
display(prediction_quality_df)


### 🔷 Step 5 — Compute Evaluation Metrics

* Compute overall multiple-choice evaluation metrics from the prepared evaluation dataset.
* Calculate prediction counts, correct predictions, incorrect predictions, and choice accuracy.
* Generate question-type performance metrics when annotation category metadata is available.
* Generate answer-choice distribution metrics for ground-truth and predicted choices.
* Prepare metric tables for reporting, visualization, and downstream comparison.


In [ ]:
# ============================================================
# Step 5: Compute Evaluation Metrics
# ============================================================

import pandas as pd

print("Computing evaluation metrics...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

if "evaluation_dataset_df" not in globals():
    raise NameError(
        "evaluation_dataset_df was not found. Run Step 3 first."
    )

if "prediction_quality_df" not in globals():
    raise NameError(
        "prediction_quality_df was not found. Run Step 4 first."
    )

# ------------------------------------------------------------
# Compute overall metrics
# ------------------------------------------------------------

total_predictions = len(evaluation_dataset_df)

correct_predictions = int(
    evaluation_dataset_df["choice_correct"].sum()
)

incorrect_predictions = (
    total_predictions - correct_predictions
)

choice_accuracy = (
    correct_predictions / total_predictions
    if total_predictions > 0
    else 0.0
)

overall_metrics_df = pd.DataFrame(
    [
        {
            "metric": "total_predictions",
            "value": total_predictions,
        },
        {
            "metric": "correct_predictions",
            "value": correct_predictions,
        },
        {
            "metric": "incorrect_predictions",
            "value": incorrect_predictions,
        },
        {
            "metric": "choice_accuracy",
            "value": choice_accuracy,
        },
    ]
)

# ------------------------------------------------------------
# Compute question-type metrics
# ------------------------------------------------------------

if "type" in evaluation_dataset_df.columns:

    question_type_metrics_df = (
        evaluation_dataset_df
        .groupby("type", dropna=False)
        .agg(
            prediction_count=("choice_correct", "size"),
            correct_predictions=("choice_correct", "sum"),
        )
        .reset_index()
    )

    question_type_metrics_df["choice_accuracy"] = (
        question_type_metrics_df["correct_predictions"] /
        question_type_metrics_df["prediction_count"]
    )

else:

    question_type_metrics_df = pd.DataFrame(
        columns=[
            "type",
            "prediction_count",
            "correct_predictions",
            "choice_accuracy",
        ]
    )

# ------------------------------------------------------------
# Compute answer-choice distribution metrics
# ------------------------------------------------------------

ground_truth_choice_distribution_df = (
    evaluation_dataset_df["ground_truth_choice"]
    .value_counts(dropna=False)
    .rename_axis("choice")
    .reset_index(name="ground_truth_count")
    .sort_values("choice")
    .reset_index(drop=True)
)

predicted_choice_distribution_df = (
    evaluation_dataset_df["predicted_choice"]
    .value_counts(dropna=False)
    .rename_axis("choice")
    .reset_index(name="predicted_count")
    .sort_values("choice")
    .reset_index(drop=True)
)

choice_distribution_df = (
    ground_truth_choice_distribution_df
    .merge(
        predicted_choice_distribution_df,
        on="choice",
        how="outer",
    )
    .fillna(0)
    .sort_values("choice")
    .reset_index(drop=True)
)

choice_distribution_df["ground_truth_count"] = (
    choice_distribution_df["ground_truth_count"].astype(int)
)

choice_distribution_df["predicted_count"] = (
    choice_distribution_df["predicted_count"].astype(int)
)

# ------------------------------------------------------------
# Display evaluation metrics
# ------------------------------------------------------------

print("Overall Evaluation Metrics")
print("-" * 60)
display(overall_metrics_df)

print("\nQuestion-Type Metrics")
print("-" * 60)
display(question_type_metrics_df)

print("\nAnswer-Choice Distribution")
print("-" * 60)
display(choice_distribution_df)



### 🔷 Step 6 — Analyze Prediction Errors

* Identify incorrect prediction records for qualitative review.
* Compare ground-truth answer choices against predicted answer choices.
* Summarize errors by question type and answer-choice pattern.
* Generate an error-analysis dataset for reporting and downstream inspection.
* Display representative incorrect predictions to support interpretation of model behavior.



In [ ]:
# ============================================================
# Step 6: Analyze Prediction Errors
# ============================================================

import pandas as pd

print("Analyzing prediction errors...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

if "evaluation_dataset_df" not in globals():
    raise NameError(
        "evaluation_dataset_df was not found. Run Step 3 first."
    )

# ------------------------------------------------------------
# Create correct and incorrect prediction datasets
# ------------------------------------------------------------

correct_predictions_df = (
    evaluation_dataset_df[
        evaluation_dataset_df["choice_correct"] == True
    ]
    .copy()
    .reset_index(drop=True)
)

incorrect_predictions_df = (
    evaluation_dataset_df[
        evaluation_dataset_df["choice_correct"] == False
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Compute error summary by question type
# ------------------------------------------------------------

if "type" in evaluation_dataset_df.columns:

    error_type_summary_df = (
        incorrect_predictions_df
        .groupby("type", dropna=False)
        .agg(
            error_count=("choice_correct", "size"),
        )
        .reset_index()
        .sort_values(
            by="error_count",
            ascending=False,
        )
        .reset_index(drop=True)
    )

else:

    error_type_summary_df = pd.DataFrame(
        columns=[
            "type",
            "error_count",
        ]
    )

# ------------------------------------------------------------
# Compute error summary by answer-choice pattern
# ------------------------------------------------------------

error_choice_pattern_df = (
    incorrect_predictions_df
    .groupby(
        [
            "ground_truth_choice",
            "predicted_choice",
        ],
        dropna=False,
    )
    .agg(
        error_count=("choice_correct", "size"),
    )
    .reset_index()
    .sort_values(
        by="error_count",
        ascending=False,
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Select display columns
# ------------------------------------------------------------

preferred_error_display_columns = [
    "video",
    "question",
    "type",
    "ground_truth_choice",
    "predicted_choice",
    "ground_truth",
    "prediction",
    "prediction_score",
]

available_error_display_columns = [
    column
    for column in preferred_error_display_columns
    if column in incorrect_predictions_df.columns
]

# ------------------------------------------------------------
# Display error analysis
# ------------------------------------------------------------

print("Prediction Error Analysis")
print("-" * 60)
print(f"Correct predictions       : {len(correct_predictions_df):,}")
print(f"Incorrect predictions     : {len(incorrect_predictions_df):,}")

print("\nErrors by Question Type")
print("-" * 60)
display(error_type_summary_df)

print("\nErrors by Choice Pattern")
print("-" * 60)
display(error_choice_pattern_df)

print("\nIncorrect Prediction Examples")
print("-" * 60)

if incorrect_predictions_df.empty:
    print("No incorrect predictions found.")
else:
    display(
        incorrect_predictions_df[
            available_error_display_columns
        ].head(10)
    )



### 🔷 Step 7 — Generate Evaluation Visualizations

* Create visual summaries of overall multiple-choice evaluation results.
* Plot question-type accuracy to compare performance across NExT-QA question categories.
* Plot ground-truth and predicted answer-choice distributions.
* Save generated figures to the evaluation output directory for reporting and documentation.
* Display generated visualizations for review inside the notebook.


In [ ]:
# ============================================================
# Step 7: Generate Evaluation Visualizations
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd

print("Generating evaluation visualizations...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

if "overall_metrics_df" not in globals():
    raise NameError(
        "overall_metrics_df was not found. Run Step 5 first."
    )

if "question_type_metrics_df" not in globals():
    raise NameError(
        "question_type_metrics_df was not found. Run Step 5 first."
    )

if "choice_distribution_df" not in globals():
    raise NameError(
        "choice_distribution_df was not found. Run Step 5 first."
    )

# ------------------------------------------------------------
# Create figure output directory
# ------------------------------------------------------------

EVALUATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

evaluation_figures_dir = (
    EVALUATION_OUTPUT_DIR / "figures"
)

evaluation_figures_dir.mkdir(
    parents=True,
    exist_ok=True,
)

generated_figures = []

# ------------------------------------------------------------
# Plot overall prediction counts
# ------------------------------------------------------------

overall_count_rows = (
    overall_metrics_df[
        overall_metrics_df["metric"].isin(
            [
                "correct_predictions",
                "incorrect_predictions",
            ]
        )
    ]
    .copy()
)

plt.figure(figsize=(6, 4))
plt.bar(
    overall_count_rows["metric"],
    overall_count_rows["value"],
)
plt.title("Overall Prediction Results")
plt.xlabel("Prediction Outcome")
plt.ylabel("Count")
plt.xticks(rotation=20)
plt.tight_layout()

overall_results_figure = (
    evaluation_figures_dir /
    "overall_prediction_results.png"
)

plt.savefig(
    overall_results_figure,
    dpi=150,
)

plt.show()

generated_figures.append(
    {
        "figure": "overall_prediction_results",
        "path": str(overall_results_figure),
    }
)

# ------------------------------------------------------------
# Plot question-type accuracy
# ------------------------------------------------------------

if not question_type_metrics_df.empty:

    question_type_plot_df = (
        question_type_metrics_df
        .copy()
        .sort_values(
            by="choice_accuracy",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    plt.figure(figsize=(8, 4))
    plt.bar(
        question_type_plot_df["type"].astype(str),
        question_type_plot_df["choice_accuracy"],
    )
    plt.title("Choice Accuracy by Question Type")
    plt.xlabel("Question Type")
    plt.ylabel("Choice Accuracy")
    plt.ylim(0, 1)
    plt.tight_layout()

    question_type_accuracy_figure = (
        evaluation_figures_dir /
        "question_type_accuracy.png"
    )

    plt.savefig(
        question_type_accuracy_figure,
        dpi=150,
    )

    plt.show()

    generated_figures.append(
        {
            "figure": "question_type_accuracy",
            "path": str(question_type_accuracy_figure),
        }
    )

# ------------------------------------------------------------
# Plot answer-choice distribution
# ------------------------------------------------------------

choice_distribution_plot_df = (
    choice_distribution_df
    .copy()
    .sort_values("choice")
    .reset_index(drop=True)
)

x_positions = range(len(choice_distribution_plot_df))
bar_width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(
    [
        position - bar_width / 2
        for position in x_positions
    ],
    choice_distribution_plot_df["ground_truth_count"],
    width=bar_width,
    label="Ground Truth",
)
plt.bar(
    [
        position + bar_width / 2
        for position in x_positions
    ],
    choice_distribution_plot_df["predicted_count"],
    width=bar_width,
    label="Predicted",
)
plt.title("Answer Choice Distribution")
plt.xlabel("Answer Choice")
plt.ylabel("Count")
plt.xticks(
    list(x_positions),
    choice_distribution_plot_df["choice"].astype(str),
)
plt.legend()
plt.tight_layout()

choice_distribution_figure = (
    evaluation_figures_dir /
    "answer_choice_distribution.png"
)

plt.savefig(
    choice_distribution_figure,
    dpi=150,
)

plt.show()

generated_figures.append(
    {
        "figure": "answer_choice_distribution",
        "path": str(choice_distribution_figure),
    }
)

# ------------------------------------------------------------
# Save generated figure inventory
# ------------------------------------------------------------

generated_figures_df = pd.DataFrame(
    generated_figures
)

generated_figures_csv = (
    evaluation_figures_dir /
    "generated_figures.csv"
)

generated_figures_df.to_csv(
    generated_figures_csv,
    index=False,
)

if not generated_figures_csv.exists():
    raise FileNotFoundError(
        f"Failed to create generated figures inventory: "
        f"{generated_figures_csv}"
    )

# ------------------------------------------------------------
# Display visualization summary
# ------------------------------------------------------------

print("Evaluation visualizations generated.")
print("-" * 60)
print(f"Figure directory : {evaluation_figures_dir}")
print(f"Figures created  : {len(generated_figures_df):,}")

display(generated_figures_df)



### 🔷 Step 8 — Save Evaluation Results

* Save the prepared evaluation dataset and computed metric tables to the local evaluation output directory.
* Save prediction quality checks, question-type metrics, answer-choice distributions, and error-analysis outputs.
* Verify that all expected evaluation artifacts are successfully written and reloadable.
* Preserve evaluation outputs in a standardized structure for comparison across baseline, CLIP, and autoencoder experiments.
* Display saved artifact paths for downstream review and reporting.



In [ ]:
# ============================================================
# Step 8: Save Evaluation Results
# ============================================================

import pandas as pd
import shutil
from pathlib import Path

print("Saving evaluation results...\n")

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_dataframes = {
    "evaluation_dataset_df": "Step 3",
    "prediction_quality_df": "Step 4",
    "overall_metrics_df": "Step 5",
    "question_type_metrics_df": "Step 5",
    "choice_distribution_df": "Step 5",
    "incorrect_predictions_df": "Step 6",
    "error_type_summary_df": "Step 6",
    "error_choice_pattern_df": "Step 6",
    "generated_figures_df": "Step 7",
}

missing_dataframes = [
    dataframe_name
    for dataframe_name in required_dataframes
    if dataframe_name not in globals()
]

if missing_dataframes:
    raise NameError(
        "Required dataframe(s) not found: "
        f"{missing_dataframes}"
    )

# ------------------------------------------------------------
# Create local evaluation output directory
# ------------------------------------------------------------

EVALUATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Define output files
# ------------------------------------------------------------

PREDICTION_QUALITY_CSV = (
    EVALUATION_OUTPUT_DIR /
    "prediction_quality_checks.csv"
)

QUESTION_TYPE_METRICS_CSV = (
    EVALUATION_OUTPUT_DIR /
    "question_type_metrics.csv"
)

CHOICE_DISTRIBUTION_CSV = (
    EVALUATION_OUTPUT_DIR /
    "answer_choice_distribution.csv"
)

INCORRECT_PREDICTIONS_CSV = (
    EVALUATION_OUTPUT_DIR /
    "incorrect_predictions.csv"
)

ERROR_TYPE_SUMMARY_CSV = (
    EVALUATION_OUTPUT_DIR /
    "error_type_summary.csv"
)

ERROR_CHOICE_PATTERN_CSV = (
    EVALUATION_OUTPUT_DIR /
    "error_choice_pattern.csv"
)

GENERATED_FIGURES_CSV = (
    EVALUATION_OUTPUT_DIR /
    "generated_figures.csv"
)

# ------------------------------------------------------------
# Save evaluation outputs
# ------------------------------------------------------------

evaluation_outputs = [
    (evaluation_dataset_df, EVALUATION_DATASET_CSV),
    (prediction_quality_df, PREDICTION_QUALITY_CSV),
    (overall_metrics_df, EVALUATION_METRICS_CSV),
    (question_type_metrics_df, QUESTION_TYPE_METRICS_CSV),
    (choice_distribution_df, CHOICE_DISTRIBUTION_CSV),
    (incorrect_predictions_df, INCORRECT_PREDICTIONS_CSV),
    (error_type_summary_df, ERROR_TYPE_SUMMARY_CSV),
    (error_choice_pattern_df, ERROR_CHOICE_PATTERN_CSV),
    (generated_figures_df, GENERATED_FIGURES_CSV),
]

for dataframe, output_path in evaluation_outputs:

    dataframe.to_csv(
        output_path,
        index=False,
    )

    if not output_path.exists():
        raise FileNotFoundError(
            f"Failed to create evaluation output file:\n"
            f"{output_path}"
        )

    reloaded_df = pd.read_csv(
        output_path
    )

    if len(reloaded_df) != len(dataframe):
        raise ValueError(
            "Saved evaluation output row count does not match "
            f"in-memory dataframe for file:\n{output_path}"
        )

# ------------------------------------------------------------
# Build local artifact inventory
# ------------------------------------------------------------

saved_evaluation_artifacts_df = pd.DataFrame(
    [
        {
            "artifact": output_path.name,
            "local_path": str(output_path),
            "record_count": len(dataframe),
        }
        for dataframe, output_path in evaluation_outputs
    ]
)

# ------------------------------------------------------------
# Promote evaluation artifacts to Google Drive
# ------------------------------------------------------------

EVALUATION_OUTPUT_DRIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

promoted_evaluation_artifacts = []

for _, artifact_row in saved_evaluation_artifacts_df.iterrows():

    local_path = Path(
        artifact_row["local_path"]
    )

    drive_path = (
        EVALUATION_OUTPUT_DRIVE_DIR /
        local_path.name
    )

    shutil.copy2(
        local_path,
        drive_path,
    )

    if not drive_path.exists():
        raise FileNotFoundError(
            f"Failed to promote evaluation artifact:\n"
            f"{drive_path}"
        )

    promoted_evaluation_artifacts.append(
        {
            "artifact": local_path.name,
            "local_path": str(local_path),
            "drive_path": str(drive_path),
            "record_count": artifact_row["record_count"],
        }
    )

promoted_evaluation_artifacts_df = pd.DataFrame(
    promoted_evaluation_artifacts
)

# ------------------------------------------------------------
# Display save summary
# ------------------------------------------------------------

print("Evaluation results saved.")
print("-" * 60)
print(f"Local output directory : {EVALUATION_OUTPUT_DIR}")
print(f"Drive output directory : {EVALUATION_OUTPUT_DRIVE_DIR}")
print(f"Artifacts saved        : {len(saved_evaluation_artifacts_df):,}")
print(f"Artifacts promoted     : {len(promoted_evaluation_artifacts_df):,}")

print("\nPromoted Evaluation Artifacts")
display(promoted_evaluation_artifacts_df)



### 🔷 Step 9 — Notebook Summary

* Summarize the completed development evaluation run.
* Report the configured evaluation source, dataset mode, split, prediction method, and representation sources.
* Display overall accuracy, prediction counts, question-type metrics, and error-analysis summary.
* List local and Google Drive output directories containing generated evaluation artifacts.
* Confirm that Notebook 08 outputs are ready for downstream comparison in Notebook 09.


In [ ]:
# ============================================================
# Step 9: Notebook Summary
# ============================================================

print("Notebook 08 complete.")
print("=" * 60)

# ------------------------------------------------------------
# Verify required inputs
# ------------------------------------------------------------

required_summary_objects = {
    "evaluation_dataset_df": "Step 3",
    "overall_metrics_df": "Step 5",
    "question_type_metrics_df": "Step 5",
    "incorrect_predictions_df": "Step 6",
    "saved_evaluation_artifacts_df": "Step 8",
    "promoted_evaluation_artifacts_df": "Step 8",
}

missing_summary_objects = [
    object_name
    for object_name in required_summary_objects
    if object_name not in globals()
]

if missing_summary_objects:
    raise NameError(
        "Required summary object(s) not found: "
        f"{missing_summary_objects}"
    )

# ------------------------------------------------------------
# Extract summary values
# ------------------------------------------------------------

total_predictions = len(
    evaluation_dataset_df
)

correct_predictions = int(
    evaluation_dataset_df["choice_correct"].sum()
)

incorrect_predictions = (
    total_predictions - correct_predictions
)

choice_accuracy = (
    correct_predictions / total_predictions
    if total_predictions > 0
    else 0.0
)

dataset_mode = "unknown"

if "prediction_summary_df" in globals():

    summary_metric_lookup = dict(
        zip(
            prediction_summary_df["metric"],
            prediction_summary_df["value"],
        )
    )

    dataset_mode = summary_metric_lookup.get(
        "dataset_mode",
        dataset_mode,
    )

    prediction_method = summary_metric_lookup.get(
        "prediction_method",
        "unknown",
    )

    video_representation_source = summary_metric_lookup.get(
        "video_representation_source",
        "unknown",
    )

    text_representation_source = summary_metric_lookup.get(
        "text_representation_source",
        "unknown",
    )

else:

    prediction_method = "unknown"
    video_representation_source = "unknown"
    text_representation_source = "unknown"

# ------------------------------------------------------------
# Display configuration summary
# ------------------------------------------------------------

print("\nDevelopment Evaluation — Configuration")
print("-" * 60)
print(f"Evaluation source          : {EVALUATION_SOURCE_NAME}")
print(f"Dataset mode               : {dataset_mode}")
print(f"Evaluation split           : {EVALUATION_SPLIT}")
print(f"Answer mode                : {ANSWER_MODE}")
print(f"Prediction method          : {prediction_method}")
print(f"Video representation       : {video_representation_source}")
print(f"Text representation        : {text_representation_source}")

# ------------------------------------------------------------
# Display evaluation summary
# ------------------------------------------------------------

print("\nEvaluation Results")
print("-" * 60)
print(f"Prediction records         : {total_predictions:,}")
print(f"Correct predictions        : {correct_predictions:,}")
print(f"Incorrect predictions      : {incorrect_predictions:,}")
print(f"Choice accuracy            : {choice_accuracy:.3f}")

# ------------------------------------------------------------
# Display question-type summary
# ------------------------------------------------------------

print("\nQuestion-Type Metrics")
print("-" * 60)

if question_type_metrics_df.empty:
    print("No question-type metrics available.")
else:
    display(question_type_metrics_df)

# ------------------------------------------------------------
# Display error summary
# ------------------------------------------------------------

print("\nError Analysis")
print("-" * 60)
print(f"Incorrect prediction rows  : {len(incorrect_predictions_df):,}")

if "error_type_summary_df" in globals() and not error_type_summary_df.empty:
    print("\nErrors by Question Type")
    display(error_type_summary_df)

# ------------------------------------------------------------
# Display output artifact summary
# ------------------------------------------------------------

print("\nOutput Artifacts")
print("-" * 60)
print(f"Local output directory     : {EVALUATION_OUTPUT_DIR}")
print(f"Drive output directory     : {EVALUATION_OUTPUT_DRIVE_DIR}")
print(f"Artifacts saved locally    : {len(saved_evaluation_artifacts_df):,}")
print(f"Artifacts promoted         : {len(promoted_evaluation_artifacts_df):,}")

print("\nNotebook 08 generated:")
print("- Evaluation dataset")
print("- Prediction quality checks")
print("- Overall evaluation metrics")
print("- Question-type metrics")
print("- Answer-choice distribution")
print("- Error-analysis reports")
print("- Evaluation visualizations")
print("- Evaluation artifact inventory")

print("\nNotebook 08 outputs are ready for downstream comparison.")

